# AI Data Analyst Agent
#
# Project Objective:
# Build an AI-powered data analyst that allows users to upload
# business datasets and ask analytical questions in natural language.
#
# The agent will select appropriate analytical tools such as:
# - Python / Pandas
# - SQL
# - Statistical analysis
# - Plotly visualization
#
# High-Level Workflow:
#
# User Question
#       ↓
# LangGraph Agent
#       ↓
# Tool Selection
#       ↓
# ┌────────────┬──────────────┬──────────────┐
# │ Pandas     │ SQL          │ Statistics   │
# │ Analysis   │ Analysis     │ Analysis     │
# └────────────┴──────────────┴──────────────┘
#                    ↓
#              Results
#                    ↓
#             LLM Interpretation
#                    ↓
#             Business Insight
#
# Main Technologies:
# Python, Pandas, NumPy, SQLite, SciPy, Statsmodels,
# Plotly, Ollama, LangChain, LangGraph and Streamlit

In [5]:
import pandas as pd
import numpy as np
import sqlite3
import scipy
import statsmodels
import plotly.express as px
print("Libraries imported successfully")

Libraries imported successfully


In [6]:
#projecet configuration

data_path = "data/superstore.csv"
db_path = "data/analyst.db"
print("Configuration loaded successfully")
print("Database path: ",db_path)
print("Dataset path: ",data_path)

Configuration loaded successfully
Database path:  data/analyst.db
Dataset path:  data/superstore.csv


In [7]:
# Load the dataset

df = pd.read_csv(data_path)

print("Dataset loaded successfully!")
print("Shape:", df.shape)
print("\nFirst 5 rows:")
display(df.head())

Dataset loaded successfully!
Shape: (10800, 21)

First 5 rows:


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2017-152156,11/8/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420.0,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2.0,0.00,41.9136
1,2,CA-2017-152156,11/8/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420.0,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3.0,0.00,219.5820
2,3,CA-2017-138688,6/12/2017,6/16/2017,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036.0,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2.0,0.00,6.8714
3,4,US-2016-108966,10/11/2016,10/18/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311.0,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5.0,0.45,-383.0310
4,5,US-2016-108966,10/11/2016,10/18/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311.0,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2.0,0.20,2.5164


In [8]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\nColumn Names:")
print(df.columns.tolist())


Rows: 10800
Columns: 21

Column Names:
['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']


In [9]:
print("Data types:")
display(df.dtypes)

print("\nMissing values:")
display(df.isnull().sum())

Data types:


,0
Row ID,object
Order ID,object
Order Date,object
Ship Date,object
Ship Mode,object
Customer ID,object
Customer Name,object
Segment,object
Country,object
City,object



Missing values:


,0
Row ID,0
Order ID,0
Order Date,806
Ship Date,806
Ship Mode,806
Customer ID,806
Customer Name,806
Segment,806
Country,806
City,806


In [10]:
print("Duplicate rows:",df.duplicated().sum())
print("\nUnique values per column: ")
display(df.nunique().sort_values())

Duplicate rows: 504

Unique values per column: 


,0
Country,1
Segment,3
Category,3
Ship Mode,4
Region,4
Discount,12
Quantity,14
Sub-Category,17
State,49
City,531


In [11]:
display(df.describe(include="all").T)

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Row ID,10800,10001,Yes,800,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Order ID,10800,5015,CA-2018-100111,28,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Order Date,9994,1236,9/5/2017,38,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Ship Date,9994,1334,12/16/2016,35,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Ship Mode,9994,4,Standard Class,5968,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Customer ID,9994,793,WB-21850,37,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Customer Name,9994,793,William Brown,37,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Segment,9994,3,Consumer,5191,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Country,9994,1,United States,9994,NaN,NaN,NaN,NaN,NaN,NaN,NaN
City,9994,531,New York City,915,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
print("Total sales:",df["Sales"].sum())
print("Total profit:",df["Profit"].sum())
print("Total quantity:",df["Quantity"].sum())
print("Average discount:",df["Discount"].sum())

Total sales: 2297200.8603
Total profit: 286397.0217
Total quantity: 37873.0
Average discount: 1561.09


In [13]:
print("Sales by category: ")
display(df.groupby("Category")["Sales"].sum().sort_values(ascending=False))
print("\nProfit by region: ")
display(df.groupby("Region")["Profit"].sum().sort_values(ascending=False))

Sales by category: 


,Sales
Category,
Technology,836154.0330
Furniture,741999.7953
Office Supplies,719047.0320



Profit by region: 


,Profit
Region,
West,108418.4489
East,91522.7800
South,46749.4303
Central,39706.3625


In [14]:
import plotly.express as px
fig=px.bar(
    df.groupby("Category",as_index=False)["Sales"].sum(),
    x="Category",
    y="Sales",
    title="Sales by Category"
)
fig.show()

In [15]:
conn=sqlite3.connect(db_path)
print("SQLite database connected successfully")

SQLite database connected successfully


In [16]:
conn.execute('DROP TABLE IF EXISTS "sales_data"')
conn.execute('DROP TABLE IF EXISTS "sales_data "')

df.to_sql("sales_data", conn, if_exists="replace", index=False)

print("Dataset loaded into SQLite successfully!")

Dataset loaded into SQLite successfully!


In [17]:
query="""
select Category,sum(Sales) as Total_Sales
from sales_data
group by Category
order by Total_Sales desc
"""
result=pd.read_sql_query(query,conn)
display(result)

,Category,Total_Sales
0,Technology,836154.0330
1,Furniture,741999.7953
2,Office Supplies,719047.0320
3,None,NaN


In [18]:
def run_sql(query):
  return pd.read_sql_query(query,conn)

  result=run_sql(
      """
select Region,sum(Profit) as Total_profit
from sales_data
group by region
order by Total_profit desc
      """
  )
  display(result)

In [19]:
def validate_sql(query):
    forbidden = ["DROP", "DELETE", "UPDATE", "INSERT", "ALTER", "CREATE"]

    if any(word in query.upper() for word in forbidden):
        raise ValueError("Only read-only SQL queries are allowed.")

    return query

print("SQL safety validation is ready!")


SQL safety validation is ready!


In [20]:
def run_pandas_analysis(operation, column):
    if operation == "sum":
        return df[column].sum()
    elif operation == "mean":
        return df[column].mean()
    elif operation == "max":
        return df[column].max()
    elif operation == "min":
        return df[column].min()
    else:
        raise ValueError("Unsupported operation")

print("Pandas analysis tool is ready!")

Pandas analysis tool is ready!


In [21]:
print("Total Sales:", run_pandas_analysis("sum", "Sales"))
print("Average Sales:", run_pandas_analysis("mean", "Sales"))
print("Maximum Sales:", run_pandas_analysis("max", "Sales"))
print("Minimum Sales:", run_pandas_analysis("min", "Sales"))

Total Sales: 2297200.8603
Average Sales: 229.85800083049827
Maximum Sales: 22638.48
Minimum Sales: 0.444


In [22]:
from scipy import stats
print("SciPy statistics module is imported successfully")

SciPy statistics module is imported successfully


In [23]:
correlation = df["Sales"].corr(df["Profit"])
print("Correlation between sales and profit: ",correlation)

Correlation between sales and profit:  0.4790643497377062


In [24]:
def correlation_test(col1,col2):
  result=stats.pearsonr(df[col1],df[col2])

  return{
      "correlation":result.statistic,
      "p_value":result.pvalue
  }
print(correlation_test("Sales","Profit"))

{'correlation': np.float64(nan), 'p_value': np.float64(nan)}


In [25]:
def create_plot(column,chart_type="bar"):
  if chart_type=="bar":
    fig=px.bar(
        df.groupby(column,as_index=False)["Sales"].sum(),
        x=column,
        y="Sales",
        title=f"Sales by {column}"
    )
  else:
    raise ValueError("UNsupported chart type")
  return fig
print("Visualization tool is ready")

Visualization tool is ready


In [26]:
fig=create_plot("Category","bar")
fig.show()

In [27]:
import os

print("LLM connection setup will be configured next.")

LLM connection setup will be configured next.


In [28]:
import requests

print("LLM integration module is ready!")

LLM integration module is ready!


In [29]:
tools = {
    "sql": run_sql,
    "pandas": run_pandas_analysis,
    "statistics": correlation_test,
    "visualization": create_plot
}

print("Agent tools registered:")
print(list(tools.keys()))

Agent tools registered:
['sql', 'pandas', 'statistics', 'visualization']


In [30]:
from typing import TypedDict, Any

class AgentState(TypedDict):
    question: str
    tool: str
    result: Any
    answer: str

print("Agent state defined successfully!")

Agent state defined successfully!


In [31]:
!apt-get update -qq
!apt-get install -y -qq zstd

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package zstd.
(Reading database ... 122809 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...


In [32]:
!curl -fsSL https://ollama.com/install.sh | sh


>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [33]:
import subprocess
import time

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(3)

print("Ollama server started!")

Ollama server started!


In [34]:
import requests

response = requests.get("http://127.0.0.1:11434/api/tags")

print("Ollama connected:", response.status_code == 200)
print(response.json())

Ollama connected: True
{'models': []}


In [35]:
!ollama pull llama3.2:3b

In [36]:
import requests

prompt = "Explain in one sentence what an AI Data Analyst Agent does."

response = requests.post(
    "http://127.0.0.1:11434/api/generate",
    json={
        "model": "llama3.2:3b",
        "prompt": prompt,
        "stream": False
    }
)

print(response.json()["response"])

An AI Data Analyst Agent is a software system that uses artificial intelligence and machine learning algorithms to analyze and interpret large datasets, providing insights and recommendations to support business decision-making.


In [37]:
!pip install -q langgraph langchain-ollama

In [38]:
from langchain_ollama import ChatOllama
llm=ChatOllama(
    model="llama3.2:3b",
    temperature=0
)
print("LangChain connected to ollama successfully")

LangChain connected to ollama successfully


In [39]:
from langchain_core.tools import tool

@tool
def sql_tool(query: str):
    """Run a read-only SQL query on the sales dataset."""
    validate_sql(query)
    return run_sql(query).to_dict(orient="records")

@tool
def pandas_tool(operation: str, column: str):
    """Perform a supported Pandas calculation."""
    return run_pandas_analysis(operation, column)

print("Agent tools created successfully!")

Agent tools created successfully!


In [40]:
llm_with_tools=llm.bind_tools([
    sql_tool,
    pandas_tool
])
print("Tools successfully bound to LLM!")

Tools successfully bound to LLM!


In [41]:
from langgraph.graph import StateGraph,START,END
class AgentState(TypedDict):
  quetion:str
  tool_result:Any
  answer:str
print("Langgraph state created successfully")

Langgraph state created successfully


In [42]:
def agent_node(state):
  response=llm_with_tools.invoke(state["question"])
  return {"tool_result":response}

print("Agent node created successfully")

Agent node created successfully


In [43]:
workflow=StateGraph(AgentState)

workflow.add_node("agent",agent_node)
workflow.add_edge(START,"agent")
workflow.add_edge("agent",END)

agent=workflow.compile()
print("LangGraph workflow compiled successfully")

LangGraph workflow compiled successfully


In [44]:
def execute_tool(state):
    response = state["tool_result"]

    if hasattr(response, "tool_calls") and response.tool_calls:
        call = response.tool_calls[0]

        if call["name"] == "sql_tool":
            result = sql_tool.invoke(call["args"])
        elif call["name"] == "pandas_tool":
            result = pandas_tool.invoke(call["args"])
        else:
            result = "Unsupported tool"

        return {"tool_result": result}

    return {"tool_result": response.content}

workflow = StateGraph(AgentState)

workflow.add_node("agent", agent_node)
workflow.add_node("tool", execute_tool)

workflow.add_edge(START, "agent")
workflow.add_edge("agent", "tool")
workflow.add_edge("tool", END)

agent = workflow.compile()

print("Agent + tool workflow compiled successfully!")

Agent + tool workflow compiled successfully!


In [45]:
from typing import TypedDict, Any
from langgraph.graph import StateGraph, START, END

class AgentState(TypedDict):
    question: str
    tool_result: Any
    answer: str


def agent_node(state):
    question = state.get("question")

    if not question:
        raise ValueError(f"Question missing from state: {state}")

    response = llm_with_tools.invoke(question)

    return {
        "question": question,
        "tool_result": response,
        "answer": ""
    }


workflow = StateGraph(AgentState)

workflow.add_node("agent", agent_node)

workflow.add_edge(START, "agent")
workflow.add_edge("agent", END)

agent = workflow.compile()

print("LangGraph state and workflow rebuilt successfully!")

LangGraph state and workflow rebuilt successfully!


In [46]:
@tool
def pandas_tool(operation: str, column: str):
    """Use only for numeric columns: Sales, Profit, Quantity, or Discount."""

    allowed_columns = ["Sales", "Profit", "Quantity", "Discount"]

    if column not in allowed_columns:
        return f"Invalid column. Choose from: {allowed_columns}"

    return run_pandas_analysis(operation, column)


llm_with_tools = llm.bind_tools([
    sql_tool,
    pandas_tool
])

print("Agent tools updated with column validation!")

Agent tools updated with column validation!


In [47]:
from langchain_core.tools import tool

@tool
def sql_tool(query: str):
    """Use for grouped analysis, filtering, sorting, comparisons, rankings, and questions involving categories, regions, or products."""
    validate_sql(query)
    return run_sql(query).to_dict(orient="records")


@tool
def pandas_tool(operation: str, column: str):
    """Use for simple calculations on one numeric column, such as sum, average, minimum, or maximum."""
    return run_pandas_analysis(operation, column)


llm_with_tools = llm.bind_tools([
    sql_tool,
    pandas_tool
])

print("Tool descriptions updated successfully!")

Tool descriptions updated successfully!


In [48]:
def sql_tool(query: str):
    """Execute a SQL query on the sales_data table."""

    if not query.strip().upper().startswith("SELECT"):
        raise ValueError("SQL tool requires a SELECT query.")

    validate_sql(query)
    return run_sql(query).to_dict(orient="records")


print("SQL tool fixed with query validation!")

SQL tool fixed with query validation!


In [49]:
query = """
SELECT Category, SUM(Sales) AS Total_Sales
FROM sales_data
GROUP BY Category
ORDER BY Total_Sales DESC
LIMIT 1
"""

result = sql_tool(query)

print("SQL tool result:")
print(result)

SQL tool result:
[{'Category': 'Technology', 'Total_Sales': 836154.033}]


In [50]:
from langchain_core.tools import tool

@tool
def sql_tool(query: str):
    """Execute a read-only SQLite SELECT query on the sales_data table.
    The query must be valid SQL, not a natural-language question."""

    if not query.strip().upper().startswith("SELECT"):
        return "ERROR: Please provide a valid SELECT SQL query."

    validate_sql(query)
    return run_sql(query).to_dict(orient="records")


llm_with_tools = llm.bind_tools([
    sql_tool,
    pandas_tool
])

print("Agent tools updated successfully!")

Agent tools updated successfully!


In [51]:
question = "Which category has the highest total sales?"

response = llm_with_tools.invoke(question)

if response.tool_calls:
    call = response.tool_calls[0]

    tool_result = sql_tool.invoke(call["args"])

    final_response = llm.invoke(
        f"""Answer the user's question using this data.

Question: {question}

Tool result: {tool_result}

Give a concise business answer."""
    )

    print(final_response.content)

Based on the provided data, the Technology category has the highest total sales, with a total of $836,154.03.


In [52]:
@tool
def pandas_tool(operation: str, column: str):
    """Use for simple calculations on numeric columns."""

    column_map = {
        "sales": "Sales",
        "profit": "Profit",
        "quantity": "Quantity",
        "discount": "Discount"
    }

    operation_map = {
        "average": "mean",
        "avg": "mean",
        "sum": "sum",
        "maximum": "max",
        "minimum": "min",
        "max": "max",
        "min": "min"
    }

    column = column_map.get(column.lower())
    operation = operation_map.get(operation.lower())

    if column is None or operation is None:
        return "Invalid column or operation."

    return run_pandas_analysis(operation, column)

print("Pandas tool improved successfully!")

Pandas tool improved successfully!


In [53]:
question = "What is the average profit?"

response = llm_with_tools.invoke(question)

print(response.tool_calls)


[{'name': 'pandas_tool', 'args': {'operation': 'average', 'column': 'profit'}, 'id': '06943926-1641-41b8-960f-7fa5174b4f04', 'type': 'tool_call'}]


In [54]:
question = "Which category has the highest sales, and what is its average profit?"

response = llm_with_tools.invoke(question)

for call in response.tool_calls:
    print("Tool:", call["name"])
    print("Arguments:", call["args"])

Tool: sql_tool
Arguments: {'query': 'SELECT category, AVG(profit) FROM sales_data GROUP BY category ORDER BY AVG(profit) DESC LIMIT 1'}


In [55]:
id="m7v2kx"
question = "Which category has the highest sales, and what is its average profit?"

response = llm_with_tools.invoke(question)

for call in response.tool_calls:
    if call["name"] == "sql_tool":
        result = sql_tool.invoke(call["args"])
        print("SQL Result:", result)

    elif call["name"] == "pandas_tool":
        result = pandas_tool.invoke(call["args"])
        print("Pandas Result:", result)

SQL Result: [{'Category': 'Technology', 'AVG(profit)': 78.75200221981592}]


In [56]:
profit_result = sql_tool.invoke({
    "query": """
    SELECT Category, AVG(Profit) AS Average_Profit
    FROM sales_data
    GROUP BY Category
    ORDER BY AVG(Profit) DESC
    LIMIT 1
    """
})

print("Highest Average Profit:", profit_result)

Highest Average Profit: [{'Category': 'Technology', 'Average_Profit': 78.75200221981592}]


In [58]:
sales_result = sql_tool.invoke({
    "query": """
    SELECT Category, SUM(Sales) AS Total_Sales
    FROM sales_data
    GROUP BY Category
    ORDER BY Total_Sales DESC
    LIMIT 1
    """
})

profit_result = sql_tool.invoke({
    "query": """
    SELECT Category, AVG(Profit) AS Average_Profit
    FROM sales_data
    GROUP BY Category
    ORDER BY Average_Profit DESC
    LIMIT 1
    """
})

print("Highest Sales:", sales_result)
print("Highest Average Profit:", profit_result)

Highest Sales: [{'Category': 'Technology', 'Total_Sales': 836154.033}]
Highest Average Profit: [{'Category': 'Technology', 'Average_Profit': 78.75200221981592}]
